In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Load Wine Dataset
wine = load_wine(as_frame=True)
df = wine.frame

# Features and Target
X = df.drop("target", axis=1)
y = df["target"]

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Baseline XGBoost Model
xgb = XGBClassifier(
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred))

# Hyperparameter Tuning
param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7]
}

grid = GridSearchCV(
    XGBClassifier(
        eval_metric="mlogloss",
        random_state=42
    ),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

best_pred = best_model.predict(X_test)

print("\nBest Parameters:")
print(grid.best_params_)

print("\nValidation Accuracy:",
      accuracy_score(y_test, best_pred))

print("\nClassification Report")
print(classification_report(y_test, best_pred))

# Training vs Testing Accuracy
train_acc = best_model.score(X_train, y_train)
test_acc = best_model.score(X_test, y_test)

print("\nTraining Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)

print("\nBest Cross Validation Score:", grid.best_score_)

print("\nTuning Decisions")
print("---------------------------")
print("Learning Rate :", grid.best_params_["learning_rate"])
print("Max Depth     :", grid.best_params_["max_depth"])
print("Estimators    :", grid.best_params_["n_estimators"])

print("\nConclusion:")
print("The tuned XGBoost model achieved the best validation performance.")